# Setup: get your lab ready for the workshop

This notebook prepares your lab in five short steps. It checks that the lab has what the
workshop needs, sets your Git name and email, and creates your workshop project. Each step
explains what it does, and its code is short enough to read.

**How to run it:** choose **Run → Run All Cells**. Each step prints `PASS` or `FAIL`.
A `FAIL` also says what to do. Fix it, then run all cells again. Running it again is safe:
it never overwrites what is already there.

The notebook changes only two things: your Git name and email, if you type them in step 4,
and your project folder `work/rrw-workshop/`, which step 5 creates if it is missing.

In [ ]:
# Tools used by the checks below, and a helper that prints one result.
import os
import subprocess
import sys
from pathlib import Path

WORK = Path.home() / "work"              # the folder that is kept when the lab stops
results = {}


def report(name, ok, detail, what_to_do):
    results[name] = ok
    print(("PASS" if ok else "FAIL") + f"  {name}: {detail}")
    if not ok:
        print(f"      What to do: {what_to_do}")

## 1. Python and the analysis packages

The workshop uses Python 3.13 with pandas, NumPy, SciPy, xarray and Matplotlib. This
cell loads them and shows their versions.

**Expected:** `PASS  Python and packages: Python 3.13.x with pandas ..., numpy ..., ...`

In [ ]:
try:
    import matplotlib, numpy, pandas, scipy, xarray
    versions = ", ".join(f"{m.__name__} {m.__version__}" for m in (pandas, numpy, scipy, xarray, matplotlib))
    ok = sys.version_info[:2] == (3, 13)
    detail = f"Python {sys.version.split()[0]} with {versions}"
except ImportError as error:
    ok, detail = False, f"a package is missing ({error.name})"

report("Python and packages", ok, detail,
       "The lab was not started from the workshop page. Stop it (File → Hub Control Panel → "
       "Stop My Server) and start it again from the workshop page.")

## 2. Example data

Reads the example dataset, *E. coli* growth at 37 °C with 1 % glucose, and plots how
the optical density of the culture rises over time.

**Expected:** `PASS  Example data: 289 readings from 8 wells`, followed by a curve that
rises from about 0.14 to about 0.58.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import pandas as pd

    data = pd.read_csv("data/ecoli-growth/ecoli-growth-37c-1pct.csv")
    wells = ["A7", "B7", "C7", "D7", "E7", "F7", "G7", "H7"]
    hours = data["tcultivo (min)"] / 60
    mean_od = data[wells].mean(axis="columns")

    plt.figure(figsize=(6, 3))
    plt.plot(hours, mean_od)
    plt.xlabel("time (h)")
    plt.ylabel("mean optical density")
    plt.title("E. coli, 37 °C, 1 % glucose")
    plt.show()

    report("Example data", len(data) == 289, f"{len(data)} readings from {len(wells)} wells",
           "The example data was changed. Stop the lab and start it again from the workshop page: "
           "the original comes back with every start.")
except (ImportError, OSError, KeyError) as error:
    report("Example data", False, f"cannot read the data ({error})",
           "Open this notebook from your home folder, the one with data/ and template/, and run it again.")

## 3. The `work/` folder is kept

Only `work/` is kept when the lab stops; everything else is reset at the next start.
This cell checks that `work/` is the kept storage and that you can write to it.

**Expected:** `PASS  work folder: /home/jovyan/work is kept storage and writable`

In [ ]:
test_file = WORK / ".setup-test.txt"
try:
    test_file.write_text("test")
    test_file.unlink()
    writable = True
except OSError:
    writable = False

kept = os.path.ismount(WORK)
report("work folder", kept and writable,
       f"{WORK} is kept storage and writable" if kept and writable
       else f"{WORK} is " + ("not writable" if kept else "not kept storage"),
       "Stop the lab (File → Hub Control Panel → Stop My Server) and start it again from the "
       "workshop page." if not kept else
       "Your storage may be full. Delete files you no longer need in work/ and run this cell again.")

## 4. Your Git name and email

Git writes your name and email into every change you record (every commit). The lab takes
both from your Jupyter4NFDI account when it starts.

**Your email goes into every commit.** Anyone can read it in a repository you publish, for
example on GitHub. To use another name or email, type it between the quotes below, for
example `EMAIL = "you@example.org"`, and run this cell again (**Shift+Enter**). A value you
set is kept in `work/` and never replaced. Leave the quotes empty to keep what is shown.

**Expected:** `PASS  Git name and email: <your name> <your email>`

In [ ]:
NAME = ""    # your name, for example "Ada Lovelace"
EMAIL = ""   # your email, for example "ada@example.org"


def git_config(key, value=""):
    if value.strip():
        subprocess.run(["git", "config", "--global", key, value.strip()], check=True)
    return subprocess.run(["git", "config", "--global", key], capture_output=True, text=True).stdout.strip()


name, email = git_config("user.name", NAME), git_config("user.email", EMAIL)
report("Git name and email", bool(name and email), f"{name or '(no name)'} <{email or 'no email'}>",
       "Type your name and email between the quotes at the top of this cell and run it again.")

## 5. Your workshop project

Your project for the whole workshop is the folder `work/rrw-workshop/`, with a Git
repository. This cell runs the setup script `setup_module.py`, which creates the folder
from `template/` and records a first commit with your name and email from step 4, so it
runs only when step 4 passed. If the folder already exists, it is left as it is. It can take up to a minute, because `work/` is
slow with many small files.

**Expected:** a line that says the script created the folder, and one that says no module
was named, which is expected: the modules add their parts during the workshop. Then
`PASS  Project: /home/jovyan/work/rrw-workshop with a Git repository and its first commit`

In [ ]:
project = WORK / "rrw-workshop"
if not results.get("Git name and email"):
    report("Project", False, "not created, because step 4 has no Git name and email yet",
           "Fix step 4 first, then run all cells again.")
else:
    setup = subprocess.run([sys.executable, str(Path.home() / "setup_module.py")], capture_output=True, text=True)
    print(setup.stdout + setup.stderr)

    history = subprocess.run(["git", "-C", str(project), "log", "--oneline"], capture_output=True, text=True)
    ok = setup.returncode == 0 and (project / "README.md").is_file() and history.stdout.strip() != ""
    report("Project", ok,
           f"{project} with a Git repository and its first commit" if ok else f"{project} is not complete",
           "Check that step 3 passed, then run all cells again. If it fails again, send the "
           "output above to your tutor.")

## Summary

**Expected:** `All 5 checks passed. Your lab is ready for the workshop.`

In [ ]:
failed = [name for name, ok in results.items() if not ok]
if failed:
    print(f"{len(failed)} of {len(results)} checks failed: {', '.join(failed)}.")
    print("Follow 'What to do' under each failed check, then run all cells again.")
else:
    print(f"All {len(results)} checks passed. Your lab is ready for the workshop.")